# FastSpeech 2 Telugu (IndicTTS Option 1B) — Google Colab GPU Training Pipeline
### Implementation of FastSpeech 2-Based Text-to-Speech Synthesis
**Dataset**: IndicTTS Telugu (Option 1B) | **Speakers**: 2 (`spk_000` Female, `spk_001` Male)
**Training Duration**: 20,000 Optimizer Updates (~140 Epochs) | **Target Loss**: Total < 1.0 (Mel < 0.25)
**Vocabulary**: 101 Graphemes | **Vocoder**: HiFi-GAN Universal (22,050 Hz)

---
### Workflow Architecture & Persistent Strategy
1. **Google Drive Persistence**: Checkpoints (`output/ckpt/IndicTTS/`) and logs (`output/log/IndicTTS/`) are continuously saved to Google Drive.
2. **Local NVMe SSD Cache**: The preprocessed feature cache (`preprocessed_data/IndicTTS/`) is extracted to `/content` for zero-latency I/O.
3. **Automated Resumption**: Training handles Colab timeouts cleanly with `--resume`, restoring step, epoch, optimizer, and loss history.
4. **Strict Test Isolation**: Test split audio, features, and transcripts are strictly isolated and never accessed during training.

## Section 1 — GPU Availability & CUDA Verification
Verifies that a GPU runtime (Tesla T4, V100, or A100) is connected and reports VRAM capacity and CUDA specifications.

In [ ]:
import torch
import sys

print("=" * 72)
print("CUDA & GPU HARDWARE VERIFICATION")
print("=" * 72)
print(f"Python Version   : {sys.version.split()[0]}")
print(f"PyTorch Version  : {torch.__version__}")

cuda_available = torch.cuda.is_available()
print(f"CUDA Available   : {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    device_props = torch.cuda.get_device_properties(0)
    vram_gb = device_props.total_memory / (1024 ** 3)
    print(f"Active GPU       : {gpu_name}")
    print(f"Total VRAM       : {vram_gb:.2f} GB")
    print(f"CUDA Capability  : {device_props.major}.{device_props.minor}")
    print(f"Device Count     : {torch.cuda.device_count()}")
    device = torch.device("cuda:0")
else:
    print("\n[WARNING] GPU not detected! Colab menu: Runtime -> Change runtime type -> T4 GPU.")
    device = torch.device("cpu")
print("=" * 72)

## Section 2 — Google Drive Integration & Directory Setup
Mounts Google Drive to persist checkpoints (`output/ckpt/IndicTTS`) and training history (`output/log/IndicTTS`).
If running locally for diagnostics, falls back to local workspace paths.

In [ ]:
import os
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/FastSpeech2_Telugu")
    PROJECT_ROOT = Path("/content/FastSpeech2_Project")
    LOCAL_CACHE_DIR = Path("/content/FastSpeech2_Project/preprocessed_data/IndicTTS")
else:
    PROJECT_ROOT = Path(".").resolve()
    DRIVE_ROOT = PROJECT_ROOT
    LOCAL_CACHE_DIR = PROJECT_ROOT / "preprocessed_data" / "IndicTTS"

DRIVE_CKPT_DIR = DRIVE_ROOT / "output" / "ckpt" / "IndicTTS"
DRIVE_LOG_DIR = DRIVE_ROOT / "output" / "log" / "IndicTTS"

DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment          : {'Google Colab' if IN_COLAB else 'Local Host'}")
print(f"Project Root         : {PROJECT_ROOT}")
print(f"Local Feature Cache  : {LOCAL_CACHE_DIR}")
print(f"Persistent Checkpoints: {DRIVE_CKPT_DIR}")
print(f"Persistent Logs      : {DRIVE_LOG_DIR}")

## Section 3 — Project Bundle & Feature Cache Extraction
Extracts `Telugu_Colab_Bundle.tar.gz` and `Telugu_IndicTTS_Features.tar.gz`.
Searches both the root directory and Google Drive for the archives.

In [ ]:
import tarfile
import shutil

def locate_archive(filename):
    search_paths = [
        Path(filename),
        Path("/content") / filename,
        DRIVE_ROOT / filename,
        DRIVE_ROOT / "archives" / filename,
    ]
    for p in search_paths:
        if p.is_file():
            return p
    return None

if IN_COLAB:
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    
    # 1. Extract Project Code Bundle
    bundle_path = locate_archive("Telugu_Colab_Bundle.tar.gz")
    if bundle_path:
        print(f"Extracting Code Bundle: {bundle_path} -> {PROJECT_ROOT}...")
        with tarfile.open(bundle_path, "r:gz") as tar:
            tar.extractall(PROJECT_ROOT)
        print("Code bundle extracted successfully.")
    else:
        print("[NOTICE] Telugu_Colab_Bundle.tar.gz not found; assuming code already mounted or in place.")

    # 2. Extract Feature Cache
    features_path = locate_archive("Telugu_IndicTTS_Features.tar.gz")
    if features_path:
        print(f"Extracting Feature Cache: {features_path} -> {PROJECT_ROOT}...")
        with tarfile.open(features_path, "r:gz") as tar:
            tar.extractall(PROJECT_ROOT)
        print("Feature cache extracted successfully.")
    else:
        print("[NOTICE] Telugu_IndicTTS_Features.tar.gz not found; assuming preprocessed_data already in place.")

# Ensure PROJECT_ROOT and original_fastspeech2 are on sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "original_fastspeech2") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "original_fastspeech2"))

print("\nProject directory setup complete.")

## Section 4 — Dependency Installation & Environment Check
Installs required packages from `requirements.txt` if missing in the Colab runtime.

In [ ]:
import importlib
import subprocess

REQUIRED_PACKAGES = [
    ("yaml", "PyYAML"),
    ("tgt", "tgt"),
    ("pyworld", "pyworld"),
    ("scipy", "scipy"),
    ("librosa", "librosa"),
    ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"),
]

missing = []
for mod, pkg in REQUIRED_PACKAGES:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing missing dependencies: {missing}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installation complete.")
else:
    print("All required dependencies are satisfied.")

## Section 5 — Dataset Integrity Validation
Validates that `preprocessed_data/IndicTTS` contains all required files, valid normalization stats, and no missing features.

In [ ]:
from Telugu.train import validate_telugu_dataset

print("Running Telugu IndicTTS Dataset Integrity Verification...")
ds_status = validate_telugu_dataset(LOCAL_CACHE_DIR)

print("=" * 72)
print(f"Dataset Status         : {ds_status['status']}")
print(f"Train Utterances       : {ds_status['train_utterances']:,} (expected 2,300)")
print(f"Validation Utterances  : {ds_status['val_utterances']:,} (expected 421)")
print(f"Speakers Verified      : {ds_status['speakers']} (expected ['spk_000', 'spk_001'])")
print(f"Feature Files Verified : {ds_status['total_features_verified']:,} (expected 16,326)")
print("=" * 72)

## Section 6 — Configuration Loading & Hyperparameter Inspection
Loads the three configuration files (`preprocess.yaml`, `model.yaml`, `train.yaml`) and verifies the configuration dataclasses.

In [ ]:
import json
import yaml
from common.config import FastSpeech2Config

PREPROCESS_YAML = PROJECT_ROOT / "config" / "IndicTTS" / "preprocess.yaml"
MODEL_YAML = PROJECT_ROOT / "config" / "IndicTTS" / "model.yaml"
TRAIN_YAML = PROJECT_ROOT / "config" / "IndicTTS" / "train.yaml"

with open(PREPROCESS_YAML, encoding="utf-8") as f:
    prep_dict = yaml.safe_load(f)
with open(MODEL_YAML, encoding="utf-8") as f:
    model_dict = yaml.safe_load(f)
with open(TRAIN_YAML, encoding="utf-8") as f:
    train_dict = yaml.safe_load(f)
with open(LOCAL_CACHE_DIR / "stats.json", encoding="utf-8") as f:
    model_dict["stats"] = json.load(f)

# Point config manifests to verified local paths
train_dict["path"]["train_manifest"] = str(LOCAL_CACHE_DIR / "train.txt")
train_dict["path"]["val_manifest"] = str(LOCAL_CACHE_DIR / "val.txt")
train_dict["path"]["ckpt_path"] = str(DRIVE_CKPT_DIR)
train_dict["path"]["log_path"] = str(DRIVE_LOG_DIR)
prep_dict["path"]["preprocessed_path"] = str(LOCAL_CACHE_DIR)

fs2_cfg = FastSpeech2Config.from_dicts(
    preprocess_dict=prep_dict,
    model_dict=model_dict,
    train_dict=train_dict,
)
prep_cfg = fs2_cfg.preprocess
model_cfg = fs2_cfg.model
train_cfg = fs2_cfg.train

print("Configuration loaded successfully:")
print(f"  Dataset Name       : {prep_cfg.dataset}")
print(f"  Language           : {prep_cfg.preprocessing.text.language}")
print(f"  Sampling Rate      : {prep_cfg.preprocessing.audio.sampling_rate} Hz")
print(f"  Mel Channels       : {prep_cfg.preprocessing.mel.n_mel_channels}")
print(f"  Grapheme Vocab     : {model_cfg.n_src_vocab} symbols")
print(f"  Speakers           : {model_cfg.n_speaker} (Multi-Speaker: {model_cfg.multi_speaker})")
print(f"  Batch Size         : {train_cfg.optimizer.batch_size} (Outer DataLoader batch: 64)")
print(f"  Warmup / Total     : {train_cfg.optimizer.warm_up_step:,} / {train_cfg.step.total_step:,} steps")

## Section 7 — Model Initialization & Parameter-Count Verification
Instantiates FastSpeech 2 on the active device and confirms that the parameter count matches the theoretical Telugu specification.

In [ ]:
from common.model.fastspeech2 import FastSpeech2

model = FastSpeech2(prep_cfg, model_cfg).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable = total_params - trainable_params

print("=" * 72)
print("FASTSPEECH 2 TELUGU MODEL ARCHITECTURE")
print("=" * 72)
print(f"Total Parameters       : {total_params:,} (expected 35,093,313)")
print(f"Trainable Parameters   : {trainable_params:,} (expected 34,580,291)")
print(f"Non-Trainable Params   : {non_trainable:,} (expected 513,022)")

va = model.variance_adaptor
print(f"Pitch Bins Boundaries  : {va.pitch_bins.shape[0]} boundaries, min={va.pitch_bins[0].item():.4f}, max={va.pitch_bins[-1].item():.4f}")
print(f"Energy Bins Boundaries : {va.energy_bins.shape[0]} boundaries, min={va.energy_bins[0].item():.4f}, max={va.energy_bins[-1].item():.4f}")

assert total_params == 35093313, f"Parameter count mismatch: got {total_params}"
assert trainable_params == 34580291, f"Trainable count mismatch: got {trainable_params}"
print("\nParameter verification PASSED.")
print("=" * 72)

## Section 8 — Hardware Smoke Test (1 Train Step + 1 Val Step)
Runs a controlled smoke test to verify forward pass, loss calculation, backward propagation, and memory allocation on the active GPU.

In [ ]:
from common.model.loss import FastSpeech2Loss
from common.training.engine import Trainer
from common.model.optimizer import ScheduledOptim
from dataset import Dataset
from torch.utils.data import DataLoader
from English.monitoring import run_model_smoke_test

loss_fn = FastSpeech2Loss(prep_cfg, model_cfg).to(device)
optimizer = ScheduledOptim(model, train_cfg, model_cfg, current_step=0)

train_dataset = Dataset(str(LOCAL_CACHE_DIR / "train.txt"), prep_dict, train_dict, sort=True, drop_last=True)
val_dataset = Dataset(str(LOCAL_CACHE_DIR / "val.txt"), prep_dict, train_dict, sort=False, drop_last=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=train_cfg.optimizer.batch_size * 4,
    shuffle=True,
    collate_fn=train_dataset.collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=train_cfg.optimizer.batch_size * 4,
    shuffle=False,
    collate_fn=val_dataset.collate_fn,
)

print("Executing GPU Model Smoke Test...")
smoke_res = run_model_smoke_test(
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
)

print("=" * 72)
print(f"Smoke Test Status : {smoke_res['status']}")
print(f"Train Total Loss  : {smoke_res['train_losses']['total']:.4f}")
print(f"Val Total Loss    : {smoke_res['val_losses']['total']:.4f}")
print("=" * 72)

## Section 9 — Launch Telugu FastSpeech 2 Training (20,000 Steps)
Runs the full training loop with:
- Checkpoints saved to Google Drive (`DRIVE_CKPT_DIR`) every 1,000 steps.
- Best model tracked and saved automatically (`checkpoint_best.pth.tar`).
- Persistent CSV logs recorded (`training_history.csv`, `validation_history.csv`).
- **Safety Gate**: Toggle `START_FULL_TRAINING = True` to begin.

In [ ]:
from Telugu.train import parse_args, run_training

# SAFETY GATE: Set to True when ready to begin GPU training
START_FULL_TRAINING = False

# Optional total steps override (e.g. 20000)
CUSTOM_TOTAL_STEPS = 20000

if START_FULL_TRAINING:
    cli_args = [
        "--preprocess_config", str(PREPROCESS_YAML),
        "--model_config", str(MODEL_YAML),
        "--train_config", str(TRAIN_YAML),
        "--preprocessed_dir", str(LOCAL_CACHE_DIR),
        "--train_manifest", str(LOCAL_CACHE_DIR / "train.txt"),
        "--val_manifest", str(LOCAL_CACHE_DIR / "val.txt"),
        "--checkpoint_dir", str(DRIVE_CKPT_DIR),
        "--log_dir", str(DRIVE_LOG_DIR),
        "--total_step", str(CUSTOM_TOTAL_STEPS),
        "--save_step", "1000",
        "--val_step", "1000",
        "--log_step", "100",
        "--max_to_keep", "5",
        "--resume",  # Automatically resumes if previous checkpoint exists on Drive
    ]
    if torch.cuda.is_available():
        cli_args.extend(["--device", "cuda"])
    else:
        cli_args.extend(["--device", "cpu", "--no_notebook_display"])

    parsed = parse_args(cli_args)
    print(f"Starting training with {parsed.total_step:,} target steps...")
    result = run_training(parsed)
    print(f"Training finished with status: {result['status']}")
else:
    print("[SAFETY GATE ACTIVE] Set START_FULL_TRAINING = True in this cell to launch training.")

## Section 10 — Checkpoint & Training Log Verification
Inspects saved checkpoints on Google Drive, confirms file integrity, and displays the loss progression curves.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from common.training.checkpoint import CheckpointManager
from English.monitoring import verify_checkpoint_file

mgr = CheckpointManager(DRIVE_CKPT_DIR)
all_ckpts = mgr.find_all_step_checkpoints()

print(f"Found {len(all_ckpts)} periodic checkpoints in {DRIVE_CKPT_DIR}:")
for step, p in all_ckpts:
    meta = verify_checkpoint_file(p)
    print(f"  Step {step:6d}: {p.name} ({meta.file_size_bytes / (1024**2):.1f} MB)")

if mgr.best_path.exists():
    best_meta = verify_checkpoint_file(mgr.best_path)
    print(f"  Best Checkpoint  : {mgr.best_path.name} (Step {best_meta.global_step}, Val Loss={best_meta.best_val_loss:.4f})")

# Display Loss Curves if log files exist
train_csv = DRIVE_LOG_DIR / "training_history.csv"
val_csv = DRIVE_LOG_DIR / "validation_history.csv"

if train_csv.exists() and val_csv.exists():
    df_train = pd.read_csv(train_csv)
    df_val = pd.read_csv(val_csv)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(df_train["global_step"], df_train["total_loss"], label="Train Total", alpha=0.7)
    axes[0].plot(df_val["global_step"], df_val["total_loss"], label="Val Total", color="red", linewidth=2)
    axes[0].set_title("Total Loss Progression")
    axes[0].set_xlabel("Steps")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True)
    axes[0].legend()
    
    axes[1].plot(df_train["global_step"], df_train["mel_loss"], label="Train Mel", alpha=0.7)
    axes[1].plot(df_val["global_step"], df_val["mel_loss"], label="Val Mel", color="orange", linewidth=2)
    axes[1].set_title("Mel-Spectrogram Loss")
    axes[1].set_xlabel("Steps")
    axes[1].grid(True)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## Section 11 — Seamless Resumption After Colab Timeout
If your Colab session disconnected, execute this cell to resume from the latest verified checkpoint on Google Drive.
The runner automatically restores the global step, learning rate schedule, and deduplicates CSV logs.

In [ ]:
# Execute this cell to resume training from the latest checkpoint on Google Drive
from Telugu.train import parse_args, run_training

resume_args = parse_args([
    "--preprocess_config", str(PREPROCESS_YAML),
    "--model_config", str(MODEL_YAML),
    "--train_config", str(TRAIN_YAML),
    "--preprocessed_dir", str(LOCAL_CACHE_DIR),
    "--checkpoint_dir", str(DRIVE_CKPT_DIR),
    "--log_dir", str(DRIVE_LOG_DIR),
    "--total_step", "20000",
    "--resume",
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
])

print("Resuming training from latest Google Drive checkpoint...")
resume_res = run_training(resume_args)
print(f"Resume run completed: {resume_res['status']}")

## Section 12 — Final Artifact Export & Packaging
Packages the best trained model checkpoint, loss plots, CSV history, and summary metadata into a downloadable archive.

In [ ]:
import tarfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
artifact_archive = DRIVE_ROOT / f"Telugu_IndicTTS_Artifacts_{timestamp}.tar.gz"

print(f"Packaging final artifacts to: {artifact_archive}...")
with tarfile.open(artifact_archive, "w:gz") as tar:
    if DRIVE_CKPT_DIR.exists():
        for f in DRIVE_CKPT_DIR.glob("*.pth.tar"):
            tar.add(f, arcname=f"checkpoints/{f.name}")
    if DRIVE_LOG_DIR.exists():
        for f in DRIVE_LOG_DIR.rglob("*"):
            if f.is_file():
                tar.add(f, arcname=f"logs/{f.relative_to(DRIVE_LOG_DIR)}")

print(f"Artifacts successfully packaged ({artifact_archive.stat().st_size / (1024**2):.2f} MB).")